# Ganga Water Quality Data Cleaning & Feature Engineering Pipeline

This notebook performs thorough data cleaning, missing value imputation, feature engineering, and normalization on the Ganga River monitoring dataset (`Datasets/stations_prepared.csv`).

### Key Steps:
1. **Dataset Loading & Inspection**: Load `stations_prepared.csv` and inspect schema, data types, and missing value distribution.
2. **Removing Unwanted Identifier Columns**: Drop non-predictive columns (`Unnamed: 0`, `station_code`, `monitoring_location`).
3. **Categorical Encoding**: Fill missing state values and apply `LabelEncoder` to `state_name`.
4. **Missing Value Imputation**: Impute missing numeric values using `KNNImputer` to preserve multi-variate correlations.
5. **Feature Engineering**: Derive parameter range features (`_range = max - min`) and parameter average features (`_avg = (max + min) / 2`).
6. **Data Export**: Save the cleaned and enriched dataset to `Datasets/cleaned_stations.csv`.
7. **Feature Standardization**: Apply `StandardScaler` to prepare normalized feature matrices for model training.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler

os.makedirs('Datasets', exist_ok=True)
os.makedirs('models', exist_ok=True)
print('Libraries imported successfully.')

Libraries imported successfully.


## 1. Load & Inspect Dataset
Load raw prepared dataset `Datasets/stations_prepared.csv`.

In [2]:
# Load raw prepared dataset
df = pd.read_csv('Datasets/stations_prepared.csv')
print(f'Initial Prepared Dataset Shape: {df.shape}')
df.head()

Initial Prepared Dataset Shape: (1426, 23)


,Unnamed: 0,station_code,monitoring_location,state_name,temp_min,temp_max,do_min,do_max,ph_min,ph_max,...,bod_max,nitrate_min,nitrate_max,fecal_coliform_min,fecal_coliform_max,total_coliform_min,total_coliform_max,fecal_streptococci_min,fecal_streptococci_max,wqi_category
0,0,1001,RIVER BEAS AT U/S MANALI,HIMACHAL PRADESH,2.0,24.0,7.8,9.2,7.2,8.2,...,2.8,0.32,1.15,2.0,170.0,63.0,540.0,2.0,2.0,Poor
1,1,2601,RIVER BEAS AT D/S MANALI,HIMACHAL PRADESH,2.0,13.0,7.6,9.0,6.5,8.1,...,2.8,0.32,1.87,110.0,1600.0,920.0,1600.0,2.0,2.0,Poor
2,2,4444,RIVER BEAS D/S OF WASTE PROCESSING FACILITY AT...,HIMACHAL PRADESH,2.0,13.0,7.8,8.8,6.7,7.8,...,2.8,0.32,1.08,110.0,1600.0,350.0,1600.0,2.0,2.0,Poor
3,3,4037,RIVER BEAS D/S MANALSU NALLAH,HIMACHAL PRADESH,2.0,14.0,7.9,8.9,6.3,8.0,...,1.0,0.32,1.74,22.0,110.0,79.0,540.0,2.0,2.0,Poor
4,4,3866,RIVER BEAS U/S BEFORE CONF. OF MANALSU NALLAH,HIMACHAL PRADESH,2.0,13.0,7.8,9.1,7.0,7.8,...,1.0,0.32,0.97,23.0,120.0,110.0,430.0,2.0,2.0,Poor


## 2. Check Missing Values
Analyze missing value counts per column.

In [3]:
# Missing values summary
missing_series = df.isnull().sum()
print('Missing values count per column:')
print(missing_series[missing_series > 0])

Missing values count per column:
state_name                 98
total_coliform_min         39
total_coliform_max         39
fecal_streptococci_min    583
fecal_streptococci_max    584
dtype: int64


## 3. Remove Non-Predictive Identifier Columns
Drop `Unnamed: 0`, `station_code`, and `monitoring_location`.

In [4]:
# Drop identifier columns
unwanted_cols = ['Unnamed: 0', 'station_code', 'monitoring_location']
df.drop(columns=[col for col in unwanted_cols if col in df.columns], inplace=True)
print(f'Removed unwanted columns: {unwanted_cols}')
print(f'Remaining columns count: {len(df.columns)}')
df.head()

Removed unwanted columns: ['Unnamed: 0', 'station_code', 'monitoring_location']
Remaining columns count: 20


,state_name,temp_min,temp_max,do_min,do_max,ph_min,ph_max,conductivity_min,conductivity_max,bod_min,bod_max,nitrate_min,nitrate_max,fecal_coliform_min,fecal_coliform_max,total_coliform_min,total_coliform_max,fecal_streptococci_min,fecal_streptococci_max,wqi_category
0,HIMACHAL PRADESH,2.0,24.0,7.8,9.2,7.2,8.2,68.0,380.0,1.0,2.8,0.32,1.15,2.0,170.0,63.0,540.0,2.0,2.0,Poor
1,HIMACHAL PRADESH,2.0,13.0,7.6,9.0,6.5,8.1,58.0,135.0,1.0,2.8,0.32,1.87,110.0,1600.0,920.0,1600.0,2.0,2.0,Poor
2,HIMACHAL PRADESH,2.0,13.0,7.8,8.8,6.7,7.8,62.0,113.0,1.0,2.8,0.32,1.08,110.0,1600.0,350.0,1600.0,2.0,2.0,Poor
3,HIMACHAL PRADESH,2.0,14.0,7.9,8.9,6.3,8.0,52.0,137.0,1.0,1.0,0.32,1.74,22.0,110.0,79.0,540.0,2.0,2.0,Poor
4,HIMACHAL PRADESH,2.0,13.0,7.8,9.1,7.0,7.8,51.0,113.0,1.0,1.0,0.32,0.97,23.0,120.0,110.0,430.0,2.0,2.0,Poor


## 4. Encode Categorical Variable (`state_name`)
Fill missing `state_name` values with `'Unknown'` and apply `LabelEncoder`.

In [5]:
# Encode categorical feature
if 'state_name' in df.columns:
    df['state_name'] = df['state_name'].fillna('Unknown')
    le = LabelEncoder()
    df['state_name'] = le.fit_transform(df['state_name'].astype(str))
    joblib.dump(le, 'models/state_label_encoder.joblib')
    print(f'Encoded state_name categories count: {len(le.classes_)}')

Encoded state_name categories count: 89


## 5. KNN Missing Value Imputation
Apply `KNNImputer(n_neighbors=5)` to fill missing numeric values (`total_coliform`, `fecal_streptococci`) based on nearest neighbor feature correlations.

In [6]:
# Separate features and target
X_raw = df.drop(columns=['wqi_category'])
y = df['wqi_category']

print(f'Total missing values before imputation: {X_raw.isnull().sum().sum()}')

# KNN Imputer
imputer = KNNImputer(n_neighbors=5)
X_imputed_array = imputer.fit_transform(X_raw)
X_imputed = pd.DataFrame(X_imputed_array, columns=X_raw.columns)
joblib.dump(imputer, 'models/knn_imputer.joblib')

print(f'Total missing values remaining after KNNImputer: {X_imputed.isnull().sum().sum()}')

Total missing values before imputation: 1245
Total missing values remaining after KNNImputer: 0


## 6. Feature Engineering
Create new domain-specific features:
- Parameter Range: `_range = max - min`
- Parameter Average: `_avg = (max + min) / 2`

In [7]:
# Feature Engineering
base_params = ['temp', 'do', 'ph', 'conductivity', 'bod', 'nitrate', 'fecal_coliform', 'total_coliform']
for base in base_params:
    if f'{base}_min' in X_imputed.columns and f'{base}_max' in X_imputed.columns:
        X_imputed[f'{base}_range'] = X_imputed[f'{base}_max'] - X_imputed[f'{base}_min']
        X_imputed[f'{base}_avg'] = (X_imputed[f'{base}_max'] + X_imputed[f'{base}_min']) / 2.0

print(f'Engineered feature set matrix shape: {X_imputed.shape}')
X_imputed.head()

Engineered feature set matrix shape: (1426, 35)


,state_name,temp_min,temp_max,do_min,do_max,ph_min,ph_max,conductivity_min,conductivity_max,bod_min,...,conductivity_range,conductivity_avg,bod_range,bod_avg,nitrate_range,nitrate_avg,fecal_coliform_range,fecal_coliform_avg,total_coliform_range,total_coliform_avg
0,31.0,2.0,24.0,7.8,9.2,7.2,8.2,68.0,380.0,1.0,...,312.0,224.0,1.8,1.9,0.83,0.735,168.0,86.0,477.0,301.5
1,31.0,2.0,13.0,7.6,9.0,6.5,8.1,58.0,135.0,1.0,...,77.0,96.5,1.8,1.9,1.55,1.095,1490.0,855.0,680.0,1260.0
2,31.0,2.0,13.0,7.8,8.8,6.7,7.8,62.0,113.0,1.0,...,51.0,87.5,1.8,1.9,0.76,0.700,1490.0,855.0,1250.0,975.0
3,31.0,2.0,14.0,7.9,8.9,6.3,8.0,52.0,137.0,1.0,...,85.0,94.5,0.0,1.0,1.42,1.030,88.0,66.0,461.0,309.5
4,31.0,2.0,13.0,7.8,9.1,7.0,7.8,51.0,113.0,1.0,...,62.0,82.0,0.0,1.0,0.65,0.645,97.0,71.5,320.0,270.0


## 7. Export Cleaned Dataset
Save cleaned dataset to `Datasets/cleaned_stations.csv`.

In [8]:
# Combine features and target
df_cleaned = X_imputed.copy()
df_cleaned['wqi_category'] = y.values
df_cleaned.to_csv('Datasets/cleaned_stations.csv', index=False)
print(f'Successfully exported cleaned dataset to Datasets/cleaned_stations.csv ({df_cleaned.shape[0]} rows, {df_cleaned.shape[1]} columns)')

Successfully exported cleaned dataset to Datasets/cleaned_stations.csv (1426 rows, 36 columns)


## 8. Feature Standardization & Scaling
Standardize feature values using `StandardScaler` and save scaler artifact to `models/dataset_scaler.joblib`.

In [9]:
# Standardization
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
joblib.dump(scaler, 'models/dataset_scaler.joblib')

print(f'Standardized feature matrix shape: {X_scaled.shape}')
print('Saved scaler to models/dataset_scaler.joblib')

Standardized feature matrix shape: (1426, 35)
Saved scaler to models/dataset_scaler.joblib
